# Session 06A: Lost in the Middle & Hybrid Search (Student Lab)
### Exploring Context Position Degeneration and Fusing BM25 with Vector Retrieval
**Foundations of Agentic AI Systems • Hands-On Laboratory Workstation**

---

### Resources & API Keys
* **Course Materials**: [Google Drive Shared Folder](https://drive.google.com/drive/folders/1qpWAHvdEZiCfM1kNCZLdhsNPl7S3DBqB?usp=sharing)
* **Free LLM API Keys**: [Awesome Free LLM APIs](https://github.com/mnfst/awesome-free-llm-apis)
* **Dual Execution Mode**: This notebook runs completely offline with local zero-cost deterministic mock models. If you have a free **Groq** or **OpenRouter** API key, you can plug it in to run against live frontier models.


## 📺 Curated 5–7 Minute Video Primers
Before starting, watch these short conceptual primers:
1. **Lost in the Middle: How Language Models Use Long Contexts** (~5 mins)  
   *Search*: `Lost in the Middle LLM paper explained Nelson Liu`  
   *Summary*: Explains the U-shaped attention degradation where facts placed in the middle of long prompts are systematically neglected.
2. **Hybrid Search: Combining BM25 and Vector Search** (~6 mins)  
   *Search*: `Hybrid Search BM25 Vector RRF Pinecone Weaviate`  
   *Summary*: Visualizes why keyword search catches exact part numbers that vector search dilutes, and how Reciprocal Rank Fusion unites them.


## 0. Environment Setup & Dual Execution Engine
This notebook uses standard scientific Python (`numpy`, `math`, `collections`). You can run it 100% offline using our built-in deterministic embedding/LLM simulator, or plug in your free **Groq** API key.


In [ ]:
import os, math, re
from collections import defaultdict, Counter
import numpy as np

# Optional: Set your Groq or OpenRouter API key
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

class MockLLMEngine:
    """Deterministic fast local simulator for zero-cost execution."""
    def generate(self, prompt: str) -> str:
        if "rephrase" in prompt.lower():
            return "Hypothetical passage discussing technical metrics and system parameters."
        return "Deterministic response based on retrieved facts."

llm = MockLLMEngine()
print("Execution environment initialized successfully!")


---
### Activity 1 (Tier 1 - Warmup): Minimal Dense Vector Store & Cosine Similarity
**Concept**: In dense retrieval, texts are mapped to unit-normalized vectors. Cosine similarity measures the angle between vectors ($S_{cos}(u, v) = u \cdot v$ when $||u||=||v||=1$).
*Note: This foundational activity is pre-solved for you below.*


In [ ]:
# Solution provided for foundational warmup
def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))
    return dot / (norm1 * norm2) if norm1 and norm2 else 0.0

# Mock document embeddings
doc_vectors = {
    "doc1_ai_memory": [0.85, 0.45, 0.10],
    "doc2_cooking":    [0.05, 0.95, 0.20],
    "doc3_os_paging":  [0.80, 0.35, 0.30]
}
query_vector = [0.90, 0.40, 0.05]

scores = {doc: cosine_similarity(query_vector, vec) for doc, vec in doc_vectors.items()}
print("Dense retrieval similarity scores:")
for doc, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {doc}: {score:.4f}")


---
### Activity 2 (Tier 1 - Warmup): Minimal BM25 Lexical Scorer
**Concept**: BM25 scores document relevance based on Term Frequency (TF) saturation and Inverse Document Frequency (IDF).
*Note: This foundational activity is pre-solved for you below.*


In [ ]:
# Solution provided for foundational warmup
class MiniBM25:
    def __init__(self, corpus: dict[str, str], k1=1.5, b=0.75):
        self.corpus = corpus
        self.k1 = k1
        self.b = b
        self.doc_lens = {doc_id: len(text.lower().split()) for doc_id, text in corpus.items()}
        self.avgdl = sum(self.doc_lens.values()) / max(len(corpus), 1)
        self.doc_freqs = defaultdict(int)
        for text in corpus.values():
            for word in set(text.lower().split()):
                self.doc_freqs[word] += 1
        self.idf = {word: math.log((len(corpus) - df + 0.5) / (df + 0.5) + 1.0) for word, df in self.doc_freqs.items()}

    def score(self, query: str) -> dict[str, float]:
        q_terms = query.lower().split()
        scores = defaultdict(float)
        for doc_id, text in self.corpus.items():
            words = text.lower().split()
            counts = Counter(words)
            dl = self.doc_lens[doc_id]
            for term in q_terms:
                if term in counts:
                    tf = counts[term]
                    idf = self.idf.get(term, 0.1)
                    denom = tf + self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))
                    scores[doc_id] += idf * ((tf * (self.k1 + 1.0)) / denom)
        return scores

sample_corpus = {
    "d1": "Agentic AI requires robust memory and context engineering.",
    "d2": "Operating systems manage physical RAM and secondary disk storage.",
    "d3": "Error code ERR-404 occurs when a memory page is missing."
}
bm25 = MiniBM25(sample_corpus)
print("BM25 Scores for 'ERR-404 memory':", dict(bm25.score("ERR-404 memory")))


---
### Activity 3 (Tier 1 - Warmup): Reciprocal Rank Fusion (RRF)
**Concept**: RRF merges ranked lists from different search engines without needing score normalization:
$$RRF(d) = \sum_{m} rac{1}{k + r_m(d)}$$
where $k=60$ is a smoothing constant and $r_m(d)$ is the 1-indexed rank of document $d$.
*Note: This foundational activity is pre-solved for you below.*


In [ ]:
# Solution provided for foundational warmup
def reciprocal_rank_fusion(dense_ranks: list[str], sparse_ranks: list[str], k: int = 60) -> dict[str, float]:
    rrf_scores = defaultdict(float)
    for rank, doc_id in enumerate(dense_ranks, start=1):
        rrf_scores[doc_id] += 1.0 / (k + rank)
    for rank, doc_id in enumerate(sparse_ranks, start=1):
        rrf_scores[doc_id] += 1.0 / (k + rank)
    return dict(sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True))

dense_ranked = ["doc1_ai_memory", "doc3_os_paging", "doc2_cooking"]
sparse_ranked = ["doc3_os_paging", "doc1_ai_memory", "doc2_cooking"]

fused = reciprocal_rank_fusion(dense_ranked, sparse_ranked)
print("Fused RRF Rankings:")
for doc, score in fused.items():
    print(f"  {doc}: {score:.5f}")


---
### Activity 4 (Tier 2 - Guided): The "Needle in a Haystack" Position Sweep
**Task**: Build a synthetic multi-paragraph haystack (e.g. 10 background paragraphs). Insert a target "needle" (e.g. `"The secret server password is Cobalt42"`) at fractional depths:
* Depth 0.0 (Beginning)
* Depth 0.5 (Middle)
* Depth 1.0 (End)
Evaluate retrieval rank or cosine similarity when searching for `"secret server password"`.

*Explanation*: Notice how middle-placement suffers from reduced relative attention/prominence in naive ranking.


In [ ]:
# [STUDENT TASK]: Implement the haystack builder and test inserting at depths 0.0, 0.5, 1.0
distractors = [f"Background paragraph {i}: Discussing general server metrics." for i in range(9)]
needle = "The secret server password is Cobalt42."

def build_haystack(depth_fraction: float) -> list[str]:
    # YOUR CODE HERE: insert needle at index int(depth_fraction * len(distractors))
    pass

# Test your implementation
for depth in [0.0, 0.5, 1.0]:
    # Print position of needle
    pass


---
### Activity 5 (Tier 2 - Guided): Chunking Strategy Comparison
**Task**: Compare Fixed Character Chunking (e.g. 100 characters with 20 overlap) vs Sentence-Boundary Chunking on a technical passage containing acronyms.
*Explanation*: Fixed chunking frequently cuts through acronyms (e.g. splitting `ERR-` from `404`), whereas sentence chunking preserves syntactic units.


In [ ]:
sample_text = "Agentic AI systems use MemGPT. Error ERR-404-B9 occurs during memory overflow. The system then pages to disk."

# [STUDENT TASK]: Implement fixed_chunking and sentence_chunking
# YOUR CODE HERE


---
### Activity 6 (Tier 2 - Guided): Hypothetical Document Embeddings (HyDE)
**Task**: Implement a zero-shot HyDE query expander. Given a short query `"Coimbatore AI Lab projects"`, generate a synthetic hypothetical answer using the LLM before retrieving.


In [ ]:
query = "Coimbatore AI Lab projects"

# [STUDENT TASK]: Generate a hypothetical passage representing what a gold answer would look like
# YOUR CODE HERE


---
### Activity 7 (Tier 3 - Exploration): Dense vs Sparse Failure Mode Stress Test
**Exploration**: Create 2 test cases:
1. A query that pure BM25 fails but Dense Vector succeeds (e.g., conceptual synonymy).
2. A query that Dense Vector fails but BM25 succeeds (e.g., specific part numbers or error codes).
Verify that Hybrid RRF successfully solves both!


In [ ]:
# [STUDENT EXPLORATION]: Test queries demonstrating where Dense fails vs BM25 fails
# YOUR CODE HERE


---
### Activity 8 (Tier 3 - Exploration): Cross-Encoder Re-Ranking Benchmark
**Exploration**: Take the top-5 candidates returned by Hybrid RRF and apply a secondary re-ranking pass. Analyze the latency-accuracy trade-off.


In [ ]:
# [STUDENT EXPLORATION]: Implement candidate re-ranking
# YOUR CODE HERE
